# Training RFM Model

## Imports

In [22]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import joblib

from notebooks.train_delivery_model import numeric_transformer

## Data Directories

In [23]:
PROCESSED_DATA_DIR= '../data/processed'
MODELS_DIR= '../models'

## Data Loading

In [24]:
df= pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'churn_training_data.csv'))

In [25]:
df.head()

,customer_unique_id,last_purchase_date,total_orders,total_spent,avg_review_score
0,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,15,879.27,5.0000
1,3e43e6105506432c953e165fb2acf44c,2018-02-27 18:36:39,9,1172.67,2.6429
2,ca77025e7201e3b30c44b472ff346268,2018-06-01 11:38:29,7,1122.72,5.0000
3,1b6c7548a2a1f9037c1fd3ddfed95f33,2018-02-14 13:22:12,7,959.01,5.0000
4,6469f99c1f9dfae7733b25662e7f1782,2018-06-28 00:43:34,7,758.83,5.0000


In [26]:
df.columns

Index(['customer_unique_id', 'last_purchase_date', 'total_orders',
       'total_spent', 'avg_review_score'],
      dtype='object')

In [27]:
df.shape

(93358, 5)

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93358 entries, 0 to 93357
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_unique_id  93358 non-null  object 
 1   last_purchase_date  93358 non-null  object 
 2   total_orders        93358 non-null  int64  
 3   total_spent         93358 non-null  float64
 4   avg_review_score    92755 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 3.6+ MB


In [29]:
df.describe()

,total_orders,total_spent,avg_review_score
count,93358.000000,93358.000000,92755.000000
mean,1.033420,165.916853,4.153360
std,0.209097,227.787005,1.280555
min,1.000000,9.590000,1.000000
25%,1.000000,63.100000,4.000000
50%,1.000000,107.890000,5.000000
75%,1.000000,183.120000,5.000000
max,15.000000,13664.080000,5.000000


In [30]:
# Converting TimeStamp to DateTime:
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'])

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93358 entries, 0 to 93357
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_unique_id  93358 non-null  object        
 1   last_purchase_date  93358 non-null  datetime64[ns]
 2   total_orders        93358 non-null  int64         
 3   total_spent         93358 non-null  float64       
 4   avg_review_score    92755 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 3.6+ MB


In [32]:
# Calculating Recency Days:
# We will use Max of last_purchase_date to act as 'today'

today= df['last_purchase_date'].max()
df['recency_days']= (today - df['last_purchase_date']).dt.days

In [33]:
df.head()

,customer_unique_id,last_purchase_date,total_orders,total_spent,avg_review_score,recency_days
0,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,15,879.27,5.0000,8
1,3e43e6105506432c953e165fb2acf44c,2018-02-27 18:36:39,9,1172.67,2.6429,182
2,ca77025e7201e3b30c44b472ff346268,2018-06-01 11:38:29,7,1122.72,5.0000,89
3,1b6c7548a2a1f9037c1fd3ddfed95f33,2018-02-14 13:22:12,7,959.01,5.0000,196
4,6469f99c1f9dfae7733b25662e7f1782,2018-06-28 00:43:34,7,758.83,5.0000,62


## Data Pre-Processing

In [34]:
# "customer_unique_id", "last_purchase_date" are not needed:
X = df[['recency_days', 'total_orders', 'total_spent', 'avg_review_score']]

In [35]:
# Defining Feature Types:
numeric_features= ['recency_days', 'total_orders', 'total_spent', 'avg_review_score']

In [36]:
# Pre-Processing Pipeline for Numeric Features:
numeric_transformer= Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

In [37]:
# Pre-Processing Pipeline:
preprocessor= ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
    ]
)

## Clustering Pipeline

In [38]:
# Initializing KMeans:
kmeans_model= KMeans(
    n_clusters= 3,
    random_state= 42,
    n_init= 10
)

In [39]:
# Final Pipeline with Data Pre-Processing and Model:
full_pipeline= Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('cluster', kmeans_model),
    ]
)

## Model Training

In [40]:
full_pipeline.fit(X)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['recency_days',
                                                   'total_orders',
                                                   'total_spent',
                                                   'avg_review_score'])])),
                ('cluster', KMeans(n_clusters=3, n_init=10, random_state=42))])

In [41]:
# Getting Cluster Assignments as new Column:
df['cluster']= full_pipeline.predict(X)

In [42]:
df.head()

,customer_unique_id,last_purchase_date,total_orders,total_spent,avg_review_score,recency_days,cluster
0,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,15,879.27,5.0000,8,2
1,3e43e6105506432c953e165fb2acf44c,2018-02-27 18:36:39,9,1172.67,2.6429,182,2
2,ca77025e7201e3b30c44b472ff346268,2018-06-01 11:38:29,7,1122.72,5.0000,89,2
3,1b6c7548a2a1f9037c1fd3ddfed95f33,2018-02-14 13:22:12,7,959.01,5.0000,196,2
4,6469f99c1f9dfae7733b25662e7f1782,2018-06-28 00:43:34,7,758.83,5.0000,62,2


In [43]:
df['cluster'].value_counts()

cluster
0    71521
1    19020
2     2817
Name: count, dtype: int64